In [1]:
import tensorflow as tf

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import Layer

In [3]:
df = pd.read_csv("Heart_disease_statlog.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,70,1,3,130,322,0,2,109,0,2.4,1,3,1,1
1,67,0,2,115,564,0,2,160,0,1.6,1,0,3,0
2,57,1,1,124,261,0,0,141,0,0.3,0,0,3,1
3,64,1,3,128,263,0,0,105,1,0.2,1,1,3,0
4,74,0,1,120,269,0,2,121,1,0.2,0,1,1,0


In [4]:
df=df.dropna()
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,70,1,3,130,322,0,2,109,0,2.4,1,3,1,1
1,67,0,2,115,564,0,2,160,0,1.6,1,0,3,0
2,57,1,1,124,261,0,0,141,0,0.3,0,0,3,1
3,64,1,3,128,263,0,0,105,1,0.2,1,1,3,0
4,74,0,1,120,269,0,2,121,1,0.2,0,1,1,0


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df.drop("target", axis=1), df["target"], test_size=0.2, random_state=42)


In [7]:
tf.keras.utils.set_random_seed(42)

nn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [8]:
nn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss = tf.keras.losses.BinaryCrossentropy(),
    metrics=[                  
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

history = nn_model.fit(
    X_train,
    y_train,
    epochs=100,
    #class_weight=class_weights,
)

history_df = pd.DataFrame(history.history)
print("Epochs run:", len(history_df))
history_df.tail()

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.3935 - auc: 0.3729 - loss: 5.5664 - precision: 0.4000 - recall: 0.6465
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5324 - auc: 0.4025 - loss: 3.4128 - precision: 0.4444 - recall: 0.0808        
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4028 - auc: 0.4045 - loss: 2.0243 - precision: 0.4062 - recall: 0.6566
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5139 - auc: 0.4597 - loss: 1.4142 - precision: 0.4444 - recall: 0.2424
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5787 - auc: 0.5941 - loss: 1.1073 - precision: 0.5308 - recall: 0.6970 
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5787 - auc: 0.6027 - loss: 1.0195 - precision: 0.5426 - recall: 0.5152
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5787 - auc: 0.6359 - loss: 0.8936 - precision: 0.5345 - recall: 0.6263
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/s

,accuracy,auc,loss,precision,recall
95,0.847222,0.900587,0.388447,0.858696,0.79798
96,0.847222,0.900803,0.387875,0.858696,0.79798
97,0.847222,0.901062,0.387955,0.858696,0.79798
98,0.847222,0.901407,0.387906,0.858696,0.79798
99,0.847222,0.901364,0.387313,0.858696,0.79798


In [9]:
class TrigonometricExpansion(Layer):
    def call(self, x):
        return tf.concat([x, tf.sin(x), tf.cos(x)], axis=-1)

In [20]:
tf.keras.utils.set_random_seed(42)

nn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    TrigonometricExpansion(),
    TrigonometricExpansion(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [21]:
nn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2),
    loss = tf.keras.losses.BinaryCrossentropy(),
    metrics=[                  
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc")
    ]
)

history = nn_model.fit(
    X_train,
    y_train,
    epochs=100,
    #class_weight=class_weights,
)

history_df = pd.DataFrame(history.history)
print("Epochs run:", len(history_df))
history_df.tail()

Epoch 1/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.4954 - auc: 0.4995 - loss: 16.7392 - precision: 0.4219 - recall: 0.2727
Epoch 2/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.4954 - auc: 0.5338 - loss: 6.8396 - precision: 0.4684 - recall: 0.7475 
Epoch 3/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6574 - auc: 0.6674 - loss: 2.6473 - precision: 0.6344 - recall: 0.5960        
Epoch 4/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7315 - auc: 0.7964 - loss: 1.4190 - precision: 0.7887 - recall: 0.5657
Epoch 5/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8056 - auc: 0.8778 - loss: 0.9980 - precision: 0.7317 - recall: 0.9091
Epoch 6/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8333 - auc: 0.9117 - loss: 0.6518 - precision: 0.8539 - recall: 0.7677
Epoch 7/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8194 - auc: 0.8846 - loss: 0.7681 - precision: 0.7941 - recall: 0.8182
Epoch 8/100
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/

,accuracy,auc,loss,precision,recall
95,0.916667,0.966977,0.229928,0.909091,0.909091
96,0.898148,0.967107,0.230878,0.888889,0.888889
97,0.893519,0.968747,0.225070,0.880000,0.888889
98,0.898148,0.968661,0.223657,0.888889,0.888889
99,0.902778,0.970560,0.215785,0.897959,0.888889
